# HW 4
Nicole Ton <br>
March 2026 <br>

To be honest, I don't know what I'm doing, Jessie. Hopefully you can parse through my code somehow. Shoutout Riddhi for letting me look at her code.

---
## Contents
* [Setup](#setup)
* [Image Analysis](#image-analysis)
* [Overscan Subtraction and Trimming](#sub-trim)
* [Bias Correction](#bias-corr)
* [Dark Analysis](#dark-analysis)
 

---
### Setup <a class="anchor" id="setup"></a>

### Import block

In [ ]:
# import block
import numpy as np
from astropy.io import fits
from matplotlib import pyplot as plt
from matplotlib import rc
%matplotlib inline
from astropy.visualization import hist
from ccdproc import ImageFileCollection
import ccdproc as ccdp
from astropy.modeling import fitting
from astropy.modeling.models import Polynomial1D
from astropy.nddata import CCDData
from datetime import datetime
from astropy.stats import sigma_clip, mad_std
from astropy.visualization import ZScaleInterval, hist
from astropy import units as u
import os

In [2]:
# import convenience plotting functions downloaded from 
# here: https://github.com/mwcraig/ccd-reduction-and-photometry-guide
phot_tutorial_dir = '/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/convenience_functions.py'
import sys
sys.path.insert(0,phot_tutorial_dir)
from convenience_functions import show_image

In [3]:
# style guide
plt.style.use('/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/guide.mplstyle')

# set a couple more default parameters for the plots below
rc('font', size=20)
rc('axes', grid=True)

In [4]:
# data directory
data_dir = "/Users/ngoc/Documents/8060/Imaging/"

# output directory
reduced_dir = "/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/reduced/"

### Read in files
Perhaps not the way you wanted us to do it, but I didn't want to deal with glob or ImageCollection.

In [5]:
# list of images
imgs = [data_dir+file for i, file in enumerate(os.listdir(data_dir)) if ".fits" in file]
imgs.sort()

# flats 5-62
flat_files = imgs[4:62].copy()

# biases 93-111
bias_files = imgs[93:112].copy()

# darks d01-15
dark_files = imgs[-15:].copy()

# PG1633 files 64-92, 130-139, 202-211, 221-246
# original folder does not contain 221-236
# skip the file that includes ".mag.1" --- I wasn't sure how to not include it in the original list comprehension
pg_files = imgs[63:86].copy() + imgs[87:94].copy() + imgs[130:140].copy() + imgs[202:212].copy() + imgs[221:-15].copy()

# NGC6823 files 153-163
ngc_files = imgs[150:161].copy()

### Get data from files
Using list comprehension, we get the data from each fits file into an array. We also make the array for the biassec and trimsec regions of each file.

In [6]:
flat_data = np.array([fits.getdata(file) for file in flat_files])
flat_biassec = np.array([np.concatenate((flat[:, 0:53], flat[:, 2101:]), axis=1) for flat in flat_data])
flat_trimsec = np.array([flat[:, 53:2101] for flat in flat_data])

bias_data = np.array([fits.getdata(file) for file in bias_files])
bias_biassec = np.array([np.concatenate((bias[:, 0:53],  bias[:, 2101:]), axis=1) for bias in bias_data])
bias_trimsec = np.array([bias[:, 53:2101] for bias in bias_data])

dark_data = np.array([fits.getdata(file) for file in dark_files])
dark_biassec = np.array([np.concatenate((dark[:, 0:53],  dark[:, 2101:]), axis=1) for dark in dark_data])
dark_trimsec = np.array([dark[:, 53:2101] for dark in dark_data])

pg_data = np.array([fits.getdata(file) for file in pg_files])
pg_biassec = np.array([np.concatenate((img[:, 0:53],  img[:, 2101:]), axis=1) for img in pg_data])
pg_trimsec = np.array([img[:, 53:2101] for img in pg_data])

ngc_data = np.array([fits.getdata(file) for file in ngc_files])
ngc_biassec = np.array([np.concatenate((img[:, 0:53],  img[:, 2101:]), axis=1) for img in ngc_data])
ngc_trimsec = np.array([img[:, 53:2101] for img in ngc_data])

### Image Analysis <a class="anchor" id="image-analysis"></a>
Here we determine the best way to combine our images.

In [ ]:
rows = np.array([100, 500, 1000, 1500])
for row in rows:
    bias_avg = []
    pg_avg = []
    ngc_avg = []

    for bias in bias_data:
        bias_row_mean = np.mean(bias[row, :])
        bias_avg.append(bias_row_mean)
    
    for img in pg_biassec:
        pg_bs_mean = np.mean(img[row, :])
        pg_avg.append(pg_bs_mean)
    
    for img in ngc_biassec:
        ngc_bs_mean = np.mean(img[row, :])
        ngc_avg.append(ngc_bs_mean)
    
    bias_time = np.arange(0, len(bias_avg), 1)
    pg_time = np.arange(0, len(pg_avg), 1)
    ngc_time = np.arange(0, len(ngc_avg), 1)

    plt.plot(bias_time, bias_avg, label="Bias Averages", color="black")
    plt.plot(pg_time, pg_avg, label="PG1633 Overscan Averages", color="blue")
    plt.plot(ngc_time, ngc_avg, label="NGC6823", color="red")
    plt.xlabel("File Order")
    plt.ylabel("Average Pix Value")
    plt.title("Average Pix Value vs. File Order (row "+str(row)+")")
    plt.legend(fontsize=8)
    plt.show()

<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_01.png" width="500" height="500">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_02.png" width="500" height="500">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_03.png" width="500" height="500">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_04.png" width="500" height="500">

I just chose 4 different rows. The plots generally look similar to each other in the sense that the bias levels don't really change, but for a science image, the bias average in the overscan region varies a lot. I would subtract from image and then combine to make the master bias.

### Overscan Subtraction and Trimming <a class="anchor" id="sub-trim"></a>
In which I figure out subtract_overscan and trim_image.

In [ ]:
# determine the model

hdu = fits.open(ngc_files[5])  # random science image
img = hdu[0].data
img_overscan = np.concatenate((img[:, 0:53], img[:, 2101:2200]), axis=1)

row_num = np.arange(0, len(img_overscan), 1)
mean_vals = []

for row in row_num:
    mean = np.mean(img_overscan[row, :])
    mean_vals.append(mean)

degree = 4  # change as needed

poly_fit_coeff = np.polyfit(row_num, mean_vals, degree)
poly_fit = np.poly1d(poly_fit_coeff)

chebyshev_fit_coeff = np.polynomial.chebyshev.chebfit(row_num, mean_vals, deg=degree)
cheb_fit_vals = np.polynomial.chebyshev.chebval(row_num, chebyshev_fit_coeff)

legendre_fit_coeff = np.polynomial.legendre.legfit(row_num, mean_vals, deg=degree)
legendre_fit_vals = np.polynomial.legendre.legval(row_num, legendre_fit_coeff)

hermite_fit_coeff = np.polynomial.hermite.hermfit(row_num, mean_vals, deg=degree)
hermite_fit_vals = np.polynomial.hermite.hermval(row_num, hermite_fit_coeff)

plt.scatter(row_num, mean_vals, marker="o", color="lightblue", label="Mean Values")
plt.plot(row_num, poly_fit(row_num), color="red", label="Polynomial Fit")
plt.plot(row_num, cheb_fit_vals, color="green", label="Chebyshev Fit")
plt.plot(row_num, legendre_fit_vals, color="purple", label="Legendre Fit")
plt.plot(row_num, hermite_fit_vals, color="blue", label="Hermite Fit")
plt.xlabel("Row Number")
plt.ylabel("Mean Pixel Value")
plt.title("Mean Pixel Value vs. Row Number")
plt.legend()
plt.show()

<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_05.png" width="500" height="500">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_06.png" width="500" height="500">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_07.png" width="500" height="500">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_08.png" width="500" height="500">

Top left is degree 1 and continuing. After degree 4, the fits don't really improve in any significant way, at least not visually. Also, the fits overlap each other so we can really only see the Hermite fit (blue line). Degree 1 is pretty poor. Degree 2 is better, but it's still off. Degree 4 flattens out kind of strangely, so I'll use degree 3.

In [9]:
# # overscan subtract + trim ONE image

# bias_img = fits.getdata(ngc_files[5])
# ccd = CCDData.read(ngc_files[5], unit=u.adu)
# overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
# overscan_ccd = CCDData(overscan, unit=ccd.unit)

# bias_img_s = ccdp.subtract_overscan(ccd, overscan=overscan_ccd, overscan_axis=1, model=Polynomial1D(degree=3)).data
# bias_img_st = ccdp.trim_image(bias_img_s[:, 53:2101], add_keyword=None)
# bias_img_t = ccdp.trim_image(bias_img[:, 53:2101], add_keyword=None)

# # original image
# vmin, vmax = ZScaleInterval().get_limits(bias_img)
# plt.imshow(bias_img, vmin=vmin, vmax=vmax)
# plt.grid(False)
# plt.xticks([])
# plt.yticks([])
# plt.title("Original Image")
# plt.show()

# # overscan subtracted image
# vmin, vmax = ZScaleInterval().get_limits(bias_img_s)
# plt.imshow(bias_img_s, vmin=vmin, vmax=vmax)
# plt.grid(False)
# plt.xticks([])
# plt.yticks([])
# plt.title("Overscan Subtracted")
# plt.show()

# # trimmed image
# vmin, vmax = ZScaleInterval().get_limits(bias_img_t)
# plt.imshow(bias_img_t, vmin=vmin, vmax=vmax)
# plt.grid(False)
# plt.xticks([])
# plt.yticks([])
# plt.title("Trimmed")
# plt.show()

# # overscan subtracted + trimmed
# vmin, vmax = ZScaleInterval().get_limits(bias_img_st)
# plt.imshow(bias_img_st, vmin=vmin, vmax=vmax)
# plt.grid(False)
# plt.xticks([])
# plt.yticks([])
# plt.title("Overscan Subtracted + Trimmed")
# plt.show()

Note: When using trim_image, it seems to only work when I have "add_keyword=None" in the parameters. Otherwise I get the error: AttributeError: 'numpy.ndarray' object has no attribute 'meta'. I don't really know what that entails, but it goes away with the add_keyword parameter, and the resulting images look fine.

In [ ]:
# # overscan subtraction + trimming for ALL images

# flats
flat_sub_arr = []
flat_sub_trim_arr = []

for flat in flat_files:
    img, header = fits.getdata(flat, header=True)
    ccd = CCDData.read(flat, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    os_ccd = CCDData(overscan, unit=ccd.unit)

    img_sub = ccdp.subtract_overscan(ccd, overscan=os_ccd, overscan_axis=1, model=Polynomial1D(degree=3)).data
    img_sub_trim = ccdp.trim_image(img_sub[:, 53:2101], add_keyword=None)

    flat_sub_arr.append(img_sub)
    flat_sub_trim_arr.append(img_sub_trim)

    # img_name = os.path.basename(flat)
    # output_path = os.path.join(reduced_dir, os.path.splitext(img_name)[0]+"_ot.fits")
    # fits.writeto(output_path, img_sub_trim, header, overwrite=True)

flat_sub_arr = np.array(flat_sub_arr)
flat_sub_trim_arr = np.array(flat_sub_trim_arr)

# bias
bias_sub_arr = []
bias_sub_trim_arr = []

for bias in bias_files:
    ccd = CCDData.read(bias, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    os_ccd = CCDData(overscan, unit=ccd.unit)

    img_sub = ccdp.subtract_overscan(ccd, overscan=os_ccd, overscan_axis=1, model=Polynomial1D(degree=3)).data
    img_sub_trim = ccdp.trim_image(img_sub[:, 53:2101], add_keyword=None)

    bias_sub_arr.append(img_sub)
    bias_sub_trim_arr.append(img_sub_trim)

    # img_name = os.path.basename(bias)
    # output_path = os.path.join(reduced_dir, os.path.splitext(img_name)[0]+"_ot.fits")
    # fits.writeto(output_path, img_sub_trim, header, overwrite=True)

bias_sub_arr = np.array(bias_sub_arr)
bias_sub_trim_arr = np.array(bias_sub_trim_arr)

# dark
dark_sub_arr = []
dark_sub_trim_arr = []

for dark in dark_files:
    ccd = CCDData.read(dark, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    os_ccd = CCDData(overscan, unit=ccd.unit)

    img_sub = ccdp.subtract_overscan(ccd, overscan=os_ccd, overscan_axis=1, model=Polynomial1D(degree=3)).data
    img_sub_trim = ccdp.trim_image(img_sub[:, 53:2101], add_keyword=None)

    dark_sub_arr.append(img_sub)
    dark_sub_trim_arr.append(img_sub_trim)

    # img_name = os.path.basename(dark)
    # output_path = os.path.join(reduced_dir, os.path.splitext(img_name)[0]+"_ot.fits")
    # fits.writeto(output_path, img_sub_trim, header, overwrite=True)

dark_sub_arr = np.array(dark_sub_arr)
dark_sub_trim_arr = np.array(dark_sub_trim_arr)

# PG1633
pg_sub_arr = []
pg_sub_trim_arr = []

for pg in pg_files:
    ccd = CCDData.read(pg, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    os_ccd = CCDData(overscan, unit=ccd.unit)

    img_sub = ccdp.subtract_overscan(ccd, overscan=os_ccd, overscan_axis=1, model=Polynomial1D(degree=3)).data
    img_sub_trim = ccdp.trim_image(img_sub[:, 53:2101], add_keyword=None)

    pg_sub_arr.append(img_sub)
    pg_sub_trim_arr.append(img_sub_trim)

    # img_name = os.path.basename(pg)
    # output_path = os.path.join(reduced_dir, os.path.splitext(img_name)[0]+"_ot.fits")
    # fits.writeto(output_path, img_sub_trim, header, overwrite=True)

pg_sub_arr = np.array(pg_sub_arr)
pg_sub_trim_arr = np.array(pg_sub_trim_arr)

# NGC6823
ngc_sub_arr = []
ngc_sub_trim_arr = []

for ngc in ngc_files:
    ccd = CCDData.read(ngc, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    os_ccd = CCDData(overscan, unit=ccd.unit)

    img_sub = ccdp.subtract_overscan(ccd, overscan=os_ccd, overscan_axis=1, model=Polynomial1D(degree=3)).data
    img_sub_trim = ccdp.trim_image(img_sub[:, 53:2101], add_keyword=None)

    ngc_sub_arr.append(img_sub)
    ngc_sub_trim_arr.append(img_sub_trim)

    # img_name = os.path.basename(ngc)
    # output_path = os.path.join(reduced_dir, os.path.splitext(img_name)[0]+"_ot.fits")
    # fits.writeto(output_path, img_sub_trim, header, overwrite=True)

ngc_sub_arr = np.array(ngc_sub_arr)
ngc_sub_trim_arr = np.array(ngc_sub_trim_arr)

### Bias Correction <a class="anchor" id="bias-corr"></a>
Let's make a master bias.

In [ ]:
# analyze bias levels

row_choice = 100  # random choice of row to look at

bias_time = np.arange(0, len(bias_sub_trim_arr), 1)
bias_mean = np.array([np.mean(bias[row_choice: ,]) for bias in bias_sub_trim_arr])
bias_med = np.array([np.median(bias[row_choice: ,]) for bias in bias_sub_trim_arr])

pg_time = np.arange(0, len(pg_sub_trim_arr), 1)
pg_mean = np.array([np.mean(pg[row_choice: ,]) for pg in pg_sub_trim_arr])
pg_med = np.array([np.median(pg[row_choice: , ]) for pg in pg_sub_trim_arr])

ngc_time = np.arange(0, len(ngc_sub_trim_arr), 1)
ngc_mean = np.array([np.mean(ngc[row_choice: ,]) for ngc in ngc_sub_trim_arr])
ngc_med = np.array([np.median(ngc[row_choice: ,]) for ngc in ngc_sub_trim_arr])

plt.plot(pg_time, pg_mean, color="blue", label="mean")
plt.plot(pg_time, pg_med, color="red", label="median")
plt.xlabel("File Number")
plt.ylabel("Pixel Level (row "+str(row_choice)+")")
plt.title("PG1633 Pixel Level Over Time")
plt.legend()
plt.show()

plt.plot(ngc_time, ngc_mean, color="blue", label="mean")
plt.plot(ngc_time, ngc_med, color="red", label="median")
plt.xlabel("File Number")
plt.ylabel("Pixel Level (row "+str(row_choice)+")")
plt.title("NGC6823 Pixel Level Over Time")
plt.legend()
plt.show()

plt.plot(bias_time, bias_mean, color="blue", label="mean")
plt.plot(bias_time, bias_med, color="red", label="median")
plt.xlabel("File Number")
plt.ylabel("Pixel Level (row "+str(row_choice)+")")
plt.title("Bias Pixel Level Over Time")
plt.legend()
plt.show()

<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_09.png" width="300" height="300">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_10.png" width="300" height="300">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_11.png" width="300" height="300">

For the two sets of science images, the mean and median follow each other closely. In the bias images, the median fluctuates while the mean is more stable. The master bias will be created by combining all the average images together.

In [ ]:
# master bias

reduced_files = [reduced_dir+file for i, file in enumerate(os.listdir(reduced_dir)) if "otz.fits" not in file]
reduced_files.sort()

bias_reduced_files = reduced_files[87:106].copy()

master_bias = ccdp.combine(bias_reduced_files, method="average", unit=u.adu)
master_bias_arr = np.array(master_bias)
output_path = os.path.join(reduced_dir, "master_bias.fits")
# fits.writeto(output_path, master_bias_arr, overwrite=True)

bias_cols = np.arange(0, master_bias_arr.shape[1], 1)
bias_master_means = np.array([np.mean(master_bias_arr[:, col]) for col in bias_cols])

pg_file = reduced_files[86]
pg_img = fits.getdata(pg_file)
pg_means = np.array([np.mean(pg_img[:, col]) for col in bias_cols])

ngc_file = reduced_files[120]
ngc_img = fits.getdata(ngc_file)
ngc_means = np.array([np.mean(ngc_img[:, col]) for col in bias_cols])

master_bias_file = output_path
master_bias_img = fits.getdata(master_bias_file)

vmin, vmax = ZScaleInterval().get_limits(master_bias_img)
plt.imshow(master_bias_img, vmin=vmin, vmax=vmax)
plt.grid(False)
plt.xticks([])
plt.yticks([])
plt.title("Master Bias")
plt.show()

plt.scatter(bias_cols, bias_master_means, color="blue", label="bias", s=1)
plt.xlabel("Column Number")
plt.ylabel("Average Pixel Value")
plt.title("Average Pixel Value vs. Column")
plt.legend(fontsize=20)
plt.show()

plt.scatter(bias_cols, pg_means, color="red", label="PG1633 "+pg_file[-12:], s=1)
plt.xlabel("Column Number")
plt.ylabel("Average Pixel Value")
plt.title("Average Pixel Value vs. Column")
plt.legend(fontsize=20)
plt.show()

plt.scatter(bias_cols, ngc_means, color="green", label="NGC6823 "+ngc_file[-12:], s=1)
plt.xlabel("Column Number")
plt.ylabel("Average Pixel Value")
plt.title("Average Pixel Value vs. Column")
plt.legend(fontsize=20)
plt.show()

<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_12.png" width="300" height="300">
<p></p>
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_13.png" width="300" height="300">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_14.png" width="300" height="300">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_15.png" width="300" height="300">

The master bias shows a normal graph. The image of PG1633 is slightly normal, and the image of NGC1683 is more vague. It seems the middle of an image (by column) still contains some level of bias. The next step is to subtract the master bias from each image.

In [ ]:
# bias correction
# shortened some of my naming conventions
# subtract_bias is a function but i'm choosing to keep everything in np arrays rather than CCDData

# flats
flat_otz_arr = []

for flat in flat_files:
    img, header = fits.getdata(flat, header=True)
    ccd = CCDData.read(flat, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    os_ccd = CCDData(overscan, unit=ccd.unit)

    img_o = ccdp.subtract_overscan(ccd, overscan=os_ccd, overscan_axis=1, model=Polynomial1D(degree=3)).data
    img_ot = ccdp.trim_image(img_o[:, 53:2101], add_keyword=None)
    img_otz = img_ot - master_bias_arr

    flat_otz_arr.append(img_otz)

    img_name = os.path.basename(flat)
    output_path = os.path.join(reduced_dir, os.path.splitext(img_name)[0]+"_otz.fits")
    fits.writeto(output_path, img_otz, header, overwrite=True)

flat_otz_arr = np.array(flat_otz_arr)

# biases
bias_otz_arr = []

for bias in bias_files:
    img, header = fits.getdata(bias, header=True)
    ccd = CCDData.read(bias, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    os_ccd = CCDData(overscan, unit=ccd.unit)

    img_o = ccdp.subtract_overscan(ccd, overscan=os_ccd, overscan_axis=1, model=Polynomial1D(degree=3)).data
    img_ot = ccdp.trim_image(img_o[:, 53:2101], add_keyword=None)
    img_otz = img_ot - master_bias_arr

    bias_otz_arr.append(img_otz)

    img_name = os.path.basename(bias)
    output_path = os.path.join(reduced_dir, os.path.splitext(img_name)[0]+"_otz.fits")
    fits.writeto(output_path, img_otz, header, overwrite=True)

bias_otz_arr = np.array(bias_otz_arr)

# darks
dark_otz_arr = []

for dark in dark_files:
    img, header = fits.getdata(dark, header=True)
    ccd = CCDData.read(dark, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    os_ccd = CCDData(overscan, unit=ccd.unit)

    img_o = ccdp.subtract_overscan(ccd, overscan=os_ccd, overscan_axis=1, model=Polynomial1D(degree=3)).data
    img_ot = ccdp.trim_image(img_o[:, 53:2101], add_keyword=None)
    img_otz = img_ot - master_bias_arr

    dark_otz_arr.append(img_otz)

    img_name = os.path.basename(dark)
    output_path = os.path.join(reduced_dir, os.path.splitext(img_name)[0]+"_otz.fits")
    fits.writeto(output_path, img_otz, header, overwrite=True)

dark_otz_arr = np.array(dark_otz_arr)

# PG1633
pg_otz_arr = []

for pg in pg_files:
    img, header = fits.getdata(pg, header=True)
    ccd = CCDData.read(pg, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    os_ccd = CCDData(overscan, unit=ccd.unit)

    img_o = ccdp.subtract_overscan(ccd, overscan=os_ccd, overscan_axis=1, model=Polynomial1D(degree=3)).data
    img_ot = ccdp.trim_image(img_o[:, 53:2101], add_keyword=None)
    img_otz = img_ot - master_bias_arr

    pg_otz_arr.append(img_otz)

    img_name = os.path.basename(pg)
    output_path = os.path.join(reduced_dir, os.path.splitext(img_name)[0]+"_otz.fits")
    fits.writeto(output_path, img_otz, header, overwrite=True)

pg_otz_arr = np.array(pg_otz_arr)

# NGC6823
ngc_otz_arr = []

for ngc in ngc_files:
    img, header = fits.getdata(ngc, header=True)
    ccd = CCDData.read(ngc, unit=u.adu)
    overscan = np.concatenate((ccd.data[:, 0:53], ccd.data[:, 2101:]), axis=1)
    os_ccd = CCDData(overscan, unit=ccd.unit)

    img_o = ccdp.subtract_overscan(ccd, overscan=os_ccd, overscan_axis=1, model=Polynomial1D(degree=3)).data
    img_ot = ccdp.trim_image(img_o[:, 53:2101], add_keyword=None)
    img_otz = img_ot - master_bias_arr

    ngc_otz_arr.append(img_otz)

    img_name = os.path.basename(ngc)
    output_path = os.path.join(reduced_dir, os.path.splitext(img_name)[0]+"_otz.fits")
    fits.writeto(output_path, img_otz, header, overwrite=True)

ngc_otz_arr = np.array(ngc_otz_arr)

### Dark Analysis <a class="anchor" id="dark-analysis"></a>

In [ ]:
# dark otz files
dark_otz_files = np.array([reduced_dir+dark for i, dark in enumerate(os.listdir(reduced_dir)) if "d0" in dark and "otz.fits" in dark])
dark_otz_files.sort()

tot_pix_arr = []
avg_pix_arr = []
exp_time_arr = []
dark_curr_arr = []
gain = 2.5

for dark in dark_otz_files:
    dark_otz_img, header = fits.getdata(dark, header=True)

    tot_pix = np.sum(dark_otz_img)
    avg_pix_med = np.median(dark_otz_img) * gain  # median to avoid cosmic rays
    exp_time = header["EXPTIME"]
    dark_curr = avg_pix_med / exp_time

    tot_pix_arr.append(tot_pix)
    avg_pix_arr.append(avg_pix_med)
    exp_time_arr.append(exp_time)
    dark_curr_arr.append(dark_curr)

tot_pix_arr = np.array(tot_pix_arr)
avg_pix_arr = np.array(avg_pix_arr)
exp_time_arr = np.array(exp_time_arr)
dark_curr_arr = np.array(dark_curr_arr)

dark_file_range = np.arange(0, len(dark_otz_files), 1)

plt.plot(dark_file_range, dark_curr_arr)
plt.xlabel("File")
plt.ylabel("Dark Current Level")
plt.title("Dark Current vs. File")
plt.show()

plt.plot(dark_file_range, avg_pix_arr)
plt.xlabel("File")
plt.ylabel("Median Pix Value")
plt.title("Median Pixel Value vs. File")
plt.show()

plt.scatter(exp_time_arr, tot_pix_arr)
plt.xlabel("Exposure Time (sec)")
plt.ylabel("Total Pixel Value")
plt.title("Total Pixel Value vs. Exposure Time")
plt.show()


<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_16.png" width="300" height="300">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_17.png" width="300" height="300">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_18.png" width="300" height="300">


The dark current level is pretty low overall. As the exposure times increase, the dark current is practically zero (at 300s). I'd say the range is somewhere from 0.01 - 0.36.

### Master Dark

In [ ]:
# comparing methods

# straight mean
dark_means = ccdp.combine(dark_otz_files, method="average", unit=u.adu)
dark_means_rms = np.sqrt(np.mean(np.array(dark_means)**2))

vmin, vmax = ZScaleInterval().get_limits(dark_means)
plt.imshow(dark_means, vmin=vmin, vmax=vmax)
plt.grid(False)
plt.xticks([])
plt.yticks([])
plt.title("Straight Mean")
plt.show()

# straight median
dark_med = ccdp.combine(dark_otz_files, method="median", unit=u.adu)
dark_med_rms = np.sqrt(np.mean(np.array(dark_med)**2))

vmin, vmax = ZScaleInterval().get_limits(dark_med)
plt.imshow(dark_med, vmin=vmin, vmax=vmax)
plt.grid(False)
plt.xticks([])
plt.yticks([])
plt.title("Straight Median")
plt.show()

# sigma clipping mean
dark_mean_sc = ccdp.combine(dark_otz_files, method="average", unit=u.adu, sigma_clip=True, sigma_clip_low_thresh=3, sigma_clip_high_thresh=3)
dark_mean_sc_rms = np.sqrt(np.mean(np.array(dark_mean_sc)**2))

vmin, vmax = ZScaleInterval().get_limits(dark_mean_sc)
plt.imshow(dark_mean_sc, vmin=vmin, vmax=vmax)
plt.grid(False)
plt.xticks([])
plt.yticks([])
plt.title(r"Mean ($3{\sigma}$-clipped)")
plt.show()

# sigma clipping median
dark_med_sc = ccdp.combine(dark_otz_files, method="median", unit=u.adu, sigma_clip=True, sigma_clip_low_thresh=3, sigma_clip_high_thresh=3)
dark_med_sc_rms = np.sqrt(np.mean(np.array(dark_med_sc)**2))

vmin, vmax = ZScaleInterval().get_limits(dark_med_sc)
plt.imshow(dark_med_sc, vmin=vmin, vmax=vmax)
plt.grid(False)
plt.xticks([])
plt.yticks([])
plt.title(r"Median ($3{\sigma}$-clipped)")
plt.show()

<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_19.png" width="275" height="275">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_20.png" width="275" height="275">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_21.png" width="275" height="275">
<img src="/Users/ngoc/Documents/8060/astr_8060_s26/work/nngocton03/notebooks/plots/hw4_plot_22.png" width="275" height="275">

<p>Straight means rms: 2.5775051372640605</p>
<p>Straight median rms: 2.33473469458659</p>
<p>Sigma clipped mean: 2.2561345357229916</p>
<p>Sigma clipped median: 2.3353302765152253</p>

Taking a straight mean of all darks is clearly the worst option as all the hot pixels can be seen still. The other three are more difficult to classify just visually. When looking at our rms values, the sigma clipped mean has the most minimized value, so it deviates the least from the data. Thus the master dark should be made by sigma clipping the mean.

In [ ]:
# master dark
master_dark = ccdp.combine(dark_otz_files, method="average", unit=u.adu, sigma_clip=True, sigma_clip_low_thresh=3, sigma_clip_high_thresh=3)
output_path = os.path.join(reduced_dir, "master_dark.fits")
master_dark_arr = np.array(master_dark)
fits.writeto(output_path, master_dark_arr, overwrite=True)

In [ ]:
master_dark_data = fits.getdata(reduced_dir+"master_dark.fits")

master_dark_med = np.median(master_dark_data)
master_dark_std = np.std(master_dark_data)
snr = master_dark_med / master_dark_std

sig_thresh = master_dark_med + (3 * master_dark_std)
tot_pix = np.sum(master_dark_data > 0)
hot_pix = np.sum(master_dark_data > sig_thresh)
tot_hot_pix = hot_pix / tot_pix

print("Total pixels:", tot_pix)
print("Significantly hot pixels:", hot_pix)
print("Percentage of hot pixels:", str(100*tot_hot_pix)+"%")
print("The noise is", str(1/snr)+"x", "larger than the average dark current level in the master dark.")

    Total pixels: 2254726
    Significantly hot pixels: 8546
    Percentage of hot pixels: 0.3790260989583657%
    The noise is 10.94004203818791x larger than the average dark current level in the master dark.

Over 8500 pixels are considered a "hot pixel" as they're above the 3-sigma threshold of the median dark currrent level. Increasing the sigma would show us how many are actually hot pixels and how many are just due to random noise. The noise being 10x higher than the dark current level shows that the dark current still contributes to the noise, albeit much smaller than read noise.

